In [ ]:
import pandas as pd

df_artists = pd.read_csv('df_artists.csv')

print(df_artists.shape)
df_artists.head()

In [ ]:
# Load google_trends_top3000.csv as google_trends_top3000.

google_trends_top3000 = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/google_trends_top3000.csv'
)

print(google_trends_top3000.shape)
google_trends_top3000.head()

# Load billboard_hot100_1958_2026.csv as billboard_hot100_songs, replacing the existing dataframe.

billboard_hot100_songs = pd.read_csv(
    '/Users/jamesemcnally/Documents/GitHub/spring-2026-hitmakers/billboard_hot100_1958_2026.csv'
)

print(billboard_hot100_songs.shape)
billboard_hot100_songs.head()



In [ ]:
# Find the first charting month and year per song from the week-by-week Billboard data,
# then add it to df_songs as first_charting_month_and_year after first_charting_year.
# Joins on title + performer (lowercased) to handle minor capitalization differences.

import pandas as pd

# Step 1: find first chart_week per title+performer
billboard_hot100_songs['chart_week'] = pd.to_datetime(billboard_hot100_songs['chart_week'])

first_chart_date = (
    billboard_hot100_songs
    .assign(
        _title    = billboard_hot100_songs['title'].str.lower().str.strip(),
        _performer = billboard_hot100_songs['performer'].str.lower().str.strip()
    )
    .groupby(['_title', '_performer'])['chart_week']
    .min()
    .reset_index()
)
first_chart_date['first_charting_month_and_year'] = (
    first_chart_date['chart_week'].dt.strftime('%B %Y')
)

# Step 2: map to df_songs via title + name (both lowercased)
df_songs['_title']     = df_songs['title'].str.lower().str.strip()
df_songs['_performer'] = df_songs['name'].str.lower().str.strip()

df_songs = df_songs.merge(
    first_chart_date[['_title', '_performer', 'first_charting_month_and_year']],
    on=['_title', '_performer'],
    how='left'
).drop(columns=['_title', '_performer'])

# Step 3: move column to after first_charting_year
cols = list(df_songs.columns)
cols.remove('first_charting_month_and_year')
cols.insert(cols.index('first_charting_year') + 1, 'first_charting_month_and_year')
df_songs = df_songs[cols]

print(df_songs['first_charting_month_and_year'].notna().sum(), '/', len(df_songs), 'songs mapped')
print(df_songs[['title', 'name', 'first_charting_year', 'first_charting_month_and_year']].head(5).to_string(index=False))


In [ ]:
# Plot Google Trends web search interest for 20 one-hit wonders vs 20 hitmakers,
# showing 3 months before through 8 months after each artist's first top-20 hit.
# Matches artists via performer_pre_normalized (df_songs) to column names before '_web' (google_trends).
# Artists are randomly selected each run.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Step 1: count top-20 songs per artist ────────────────────────────────────

top20_counts = (
    df_songs[df_songs['peak_pos'] <= 20]
    .groupby('performer_pre_normalized')
    .size()
    .reset_index(name='n_top20_songs')
)

# ── Step 2: get each artist's first top-20 hit ───────────────────────────────

first_top20 = (
    df_songs[df_songs['peak_pos'] <= 20]
    .dropna(subset=['performer_pre_normalized', 'first_charting_month_and_year'])
    .sort_values(['performer_pre_normalized', 'first_charting_year', 'peak_pos'])
    .drop_duplicates(subset=['performer_pre_normalized'], keep='first')
)[['performer_pre_normalized', 'first_charting_year', 'first_charting_month_and_year', 'title']]

first_top20 = first_top20.merge(top20_counts, on='performer_pre_normalized', how='left')
first_top20 = first_top20[first_top20['first_charting_year'].between(2005, 2020)]
first_top20['is_one_hit_wonder'] = first_top20['n_top20_songs'] == 1

# ── Step 3: match to google_trends _web columns ──────────────────────────────

web_cols = [c for c in google_trends_top3000.columns if c.endswith('_web')]
gt_map   = {c.replace('_web', '').lower().strip(): c for c in web_cols}

first_top20['_perf_lower'] = first_top20['performer_pre_normalized'].str.lower().str.strip()
first_top20['gt_col']      = first_top20['_perf_lower'].map(gt_map)

# ── Step 4: parse dates, check window coverage ───────────────────────────────

google_trends_top3000['_date'] = pd.to_datetime(google_trends_top3000['date'])
gt_min = google_trends_top3000['_date'].min()
gt_max = google_trends_top3000['_date'].max()

valid = first_top20[first_top20['gt_col'].notna()].copy()
valid['_debut'] = pd.to_datetime(valid['first_charting_month_and_year'], format='%B %Y')
valid = valid[
    (valid['_debut'] - pd.DateOffset(months=3) >= gt_min) &
    (valid['_debut'] + pd.DateOffset(months=8) <= gt_max)
]

# ── Step 5: randomly sample 20 from each group ───────────────────────────────

ohw       = valid[valid['is_one_hit_wonder']].sample(20)
hitmakers = valid[~valid['is_one_hit_wonder']].sample(20)

print("One-hit wonders selected:", ohw['performer_pre_normalized'].tolist())
print("Hitmakers selected:",       hitmakers['performer_pre_normalized'].tolist())

# ── Step 6: plot ──────────────────────────────────────────────────────────────

def plot_trends(group_df, title):
    fig, ax = plt.subplots(figsize=(14, 6))
    for _, row in group_df.iterrows():
        start = row['_debut'] - pd.DateOffset(months=3)
        end   = row['_debut'] + pd.DateOffset(months=8)
        mask  = (google_trends_top3000['_date'] >= start) & (google_trends_top3000['_date'] <= end)
        window = google_trends_top3000[mask][['_date', row['gt_col']]].copy()
        window[row['gt_col']] = pd.to_numeric(window[row['gt_col']], errors='coerce')
        window['month'] = window['_date'].apply(
            lambda d: (d.year - row['_debut'].year) * 12 + (d.month - row['_debut'].month)
        )
        label = f"{row['performer_pre_normalized']} ({row['first_charting_month_and_year']})"
        ax.plot(window['month'], window[row['gt_col']], marker='o', alpha=0.7, label=label)

    ax.axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.6, label='First top-20 debut')
    ax.set_xticks(range(-3, 9))
    ax.set_xticklabels([f"M{m:+d}" if m != 0 else "Debut" for m in range(-3, 9)])
    ax.set_xlabel('Months relative to first top-20 hit')
    ax.set_ylabel('Google Trends — web search interest')
    ax.set_title(title)
    ax.legend(fontsize=6, ncol=2)
    plt.tight_layout()
    plt.show()

plot_trends(ohw,       'Google Trends: One-Hit Wonders — First Top-20 Hit (2005–2020)')
plot_trends(hitmakers, 'Google Trends: Hitmakers — First Top-20 Hit (2005–2020)')


In [ ]:
# Plot Google Trends web search interest for 20 one-hit wonders vs 20 hitmakers,
# showing 3 months before through 12 months after each artist's first top-20 hit.
# Matches artists via performer_pre_normalized (df_songs) to column names before '_web' (google_trends).
# Artists are randomly selected each run.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Step 1: count top-20 songs per artist ────────────────────────────────────

top20_counts = (
    df_songs[df_songs['peak_pos'] <= 20]
    .groupby('performer_pre_normalized')
    .size()
    .reset_index(name='n_top20_songs')
)

# ── Step 2: get each artist's first top-20 hit ───────────────────────────────

first_top20 = (
    df_songs[df_songs['peak_pos'] <= 20]
    .dropna(subset=['performer_pre_normalized', 'first_charting_month_and_year'])
    .sort_values(['performer_pre_normalized', 'first_charting_year', 'peak_pos'])
    .drop_duplicates(subset=['performer_pre_normalized'], keep='first')
)[['performer_pre_normalized', 'first_charting_year', 'first_charting_month_and_year', 'title']]

first_top20 = first_top20.merge(top20_counts, on='performer_pre_normalized', how='left')
first_top20 = first_top20[first_top20['first_charting_year'].between(2005, 2020)]
first_top20['is_one_hit_wonder'] = first_top20['n_top20_songs'] == 1

# ── Step 3: match to google_trends _web columns ──────────────────────────────

web_cols = [c for c in google_trends_top3000.columns if c.endswith('_web')]
gt_map   = {c.replace('_web', '').lower().strip(): c for c in web_cols}

first_top20['_perf_lower'] = first_top20['performer_pre_normalized'].str.lower().str.strip()
first_top20['gt_col']      = first_top20['_perf_lower'].map(gt_map)

# ── Step 4: parse dates, check window coverage ───────────────────────────────

google_trends_top3000['_date'] = pd.to_datetime(google_trends_top3000['date'])
gt_min = google_trends_top3000['_date'].min()
gt_max = google_trends_top3000['_date'].max()

valid = first_top20[first_top20['gt_col'].notna()].copy()
valid['_debut'] = pd.to_datetime(valid['first_charting_month_and_year'], format='%B %Y')
valid = valid[
    (valid['_debut'] - pd.DateOffset(months=3)  >= gt_min) &
    (valid['_debut'] + pd.DateOffset(months=12) <= gt_max)
]

# ── Step 5: randomly sample 20 from each group ───────────────────────────────

ohw       = valid[valid['is_one_hit_wonder']].sample(20)
hitmakers = valid[~valid['is_one_hit_wonder']].sample(20)

print("One-hit wonders selected:", ohw['performer_pre_normalized'].tolist())
print("Hitmakers selected:",       hitmakers['performer_pre_normalized'].tolist())

# ── Step 6: plot ──────────────────────────────────────────────────────────────

def plot_trends(group_df, title):
    fig, ax = plt.subplots(figsize=(14, 6))
    for _, row in group_df.iterrows():
        start = row['_debut'] - pd.DateOffset(months=3)
        end   = row['_debut'] + pd.DateOffset(months=12)
        mask  = (google_trends_top3000['_date'] >= start) & (google_trends_top3000['_date'] <= end)
        window = google_trends_top3000[mask][['_date', row['gt_col']]].copy()
        window[row['gt_col']] = pd.to_numeric(window[row['gt_col']], errors='coerce')
        window['month'] = window['_date'].apply(
            lambda d: (d.year - row['_debut'].year) * 12 + (d.month - row['_debut'].month)
        )
        label = f"{row['performer_pre_normalized']} ({row['first_charting_month_and_year']})"
        ax.plot(window['month'], window[row['gt_col']], marker='o', alpha=0.7, label=label)

    ax.axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.6, label='First top-20 debut')
    ax.set_xticks(range(-3, 13))
    ax.set_xticklabels([f"M{m:+d}" if m != 0 else "Debut" for m in range(-3, 13)])
    ax.set_xlabel('Months relative to first top-20 hit')
    ax.set_ylabel('Google Trends — web search interest')
    ax.set_title(title)
    ax.legend(fontsize=6, ncol=2)
    plt.tight_layout()
    plt.show()

plot_trends(ohw,       'Google Trends: One-Hit Wonders — First Top-20 Hit (2005–2020)')
plot_trends(hitmakers, 'Google Trends: Hitmakers — First Top-20 Hit (2005–2020)')


In [ ]:
# Plot Google Trends YouTube search interest for 20 one-hit wonders vs 20 hitmakers,
# showing 3 months before through 12 months after each artist's first top-20 hit.
# Matches artists via performer_pre_normalized (df_songs) to column names before '_youtube' (google_trends).
# Artists are randomly selected each run.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Step 1: count top-20 songs per artist ────────────────────────────────────

top20_counts = (
    df_songs[df_songs['peak_pos'] <= 20]
    .groupby('performer_pre_normalized')
    .size()
    .reset_index(name='n_top20_songs')
)

# ── Step 2: get each artist's first top-20 hit ───────────────────────────────

first_top20 = (
    df_songs[df_songs['peak_pos'] <= 20]
    .dropna(subset=['performer_pre_normalized', 'first_charting_month_and_year'])
    .sort_values(['performer_pre_normalized', 'first_charting_year', 'peak_pos'])
    .drop_duplicates(subset=['performer_pre_normalized'], keep='first')
)[['performer_pre_normalized', 'first_charting_year', 'first_charting_month_and_year', 'title']]

first_top20 = first_top20.merge(top20_counts, on='performer_pre_normalized', how='left')
first_top20 = first_top20[first_top20['first_charting_year'].between(2005, 2020)]
first_top20['is_one_hit_wonder'] = first_top20['n_top20_songs'] == 1
# ── Step 3: match to google_trends _youtube columns ──────────────────────────

youtube_cols = [c for c in google_trends_top3000.columns if c.endswith('_youtube')]
gt_map       = {c.replace('_youtube', '').lower().strip(): c for c in youtube_cols}

first_top20['_perf_lower'] = first_top20['performer_pre_normalized'].str.lower().str.strip()
first_top20['gt_col']      = first_top20['_perf_lower'].map(gt_map)

# ── Step 4: parse dates, check window coverage ───────────────────────────────

google_trends_top3000['_date'] = pd.to_datetime(google_trends_top3000['date'])
gt_min = google_trends_top3000['_date'].min()
gt_max = google_trends_top3000['_date'].max()

valid = first_top20[first_top20['gt_col'].notna()].copy()
valid['_debut'] = pd.to_datetime(valid['first_charting_month_and_year'], format='%B %Y')
valid = valid[
    (valid['_debut'] - pd.DateOffset(months=3)  >= gt_min) &
    (valid['_debut'] + pd.DateOffset(months=12) <= gt_max)
]

# ── Step 5: randomly sample 20 from each group ───────────────────────────────

ohw       = valid[valid['is_one_hit_wonder']].sample(20)
hitmakers = valid[~valid['is_one_hit_wonder']].sample(20)

print("One-hit wonders selected:", ohw['performer_pre_normalized'].tolist())
print("Hitmakers selected:",       hitmakers['performer_pre_normalized'].tolist())

# ── Step 6: plot ──────────────────────────────────────────────────────────────

def plot_trends(group_df, title):
    fig, ax = plt.subplots(figsize=(14, 6))
    for _, row in group_df.iterrows():
        start = row['_debut'] - pd.DateOffset(months=3)
        end   = row['_debut'] + pd.DateOffset(months=12)
        mask  = (google_trends_top3000['_date'] >= start) & (google_trends_top3000['_date'] <= end)
        window = google_trends_top3000[mask][['_date', row['gt_col']]].copy()
        window[row['gt_col']] = pd.to_numeric(window[row['gt_col']], errors='coerce')
        window['month'] = window['_date'].apply(
            lambda d: (d.year - row['_debut'].year) * 12 + (d.month - row['_debut'].month)
        )
        label = f"{row['performer_pre_normalized']} ({row['first_charting_month_and_year']})"
        ax.plot(window['month'], window[row['gt_col']], marker='o', alpha=0.7, label=label)

    ax.axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.6, label='First top-20 debut')
    ax.set_xticks(range(-3, 13))
    ax.set_xticklabels([f"M{m:+d}" if m != 0 else "Debut" for m in range(-3, 13)])
    ax.set_xlabel('Months relative to first top-20 hit')
    ax.set_ylabel('Google Trends — YouTube search interest')
    ax.set_title(title)
    ax.legend(fontsize=6, ncol=2)
    plt.tight_layout()
    plt.show()

plot_trends(ohw,       'Google Trends (YouTube): One-Hit Wonders — First Top-20 Hit (2005–2020)')
plot_trends(hitmakers, 'Google Trends (YouTube): Hitmakers — First Top-20 Hit (2005–2020)')


In [ ]:
# Plot Google Trends web and YouTube interest for ALL one-hit wonders and hitmakers
# with a first top-20 hit between 2005–2020 who appear in google_trends_top3000.
# 3 months before through 12 months after debut. 4 graphs total. No artist labels.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Step 1: count top-20 songs per artist ────────────────────────────────────

top20_counts = (
    df_songs[df_songs['peak_pos'] <= 20]
    .groupby('performer_pre_normalized')
    .size()
    .reset_index(name='n_top20_songs')
)

# ── Step 2: get each artist's first top-20 hit ───────────────────────────────

first_top20 = (
    df_songs[df_songs['peak_pos'] <= 20]
    .dropna(subset=['performer_pre_normalized', 'first_charting_month_and_year'])
    .sort_values(['performer_pre_normalized', 'first_charting_year', 'peak_pos'])
    .drop_duplicates(subset=['performer_pre_normalized'], keep='first')
)[['performer_pre_normalized', 'first_charting_year', 'first_charting_month_and_year', 'title']]

first_top20 = first_top20.merge(top20_counts, on='performer_pre_normalized', how='left')
first_top20 = first_top20[first_top20['first_charting_year'].between(2005, 2020)]
first_top20['is_one_hit_wonder'] = first_top20['n_top20_songs'] == 1
first_top20['_perf_lower'] = first_top20['performer_pre_normalized'].str.lower().str.strip()

# ── Step 3: match to _web and _youtube columns ───────────────────────────────

google_trends_top3000['_date'] = pd.to_datetime(google_trends_top3000['date'])
gt_min = google_trends_top3000['_date'].min()
gt_max = google_trends_top3000['_date'].max()

for suffix in ['_web', '_youtube']:
    cols   = [c for c in google_trends_top3000.columns if c.endswith(suffix)]
    gt_map = {c.replace(suffix, '').lower().strip(): c for c in cols}
    first_top20[f'gt_col{suffix}'] = first_top20['_perf_lower'].map(gt_map)

first_top20['_debut'] = pd.to_datetime(first_top20['first_charting_month_and_year'], format='%B %Y')
first_top20 = first_top20[
    (first_top20['_debut'] - pd.DateOffset(months=3)  >= gt_min) &
    (first_top20['_debut'] + pd.DateOffset(months=12) <= gt_max)
]

# ── Step 4: plot function ─────────────────────────────────────────────────────

def plot_trends(group_df, gt_col_field, ylabel, title):
    sub = group_df[group_df[gt_col_field].notna()]
    print(f"  {title}: {len(sub)} artists")
    fig, ax = plt.subplots(figsize=(14, 6))
    for _, row in sub.iterrows():
        start = row['_debut'] - pd.DateOffset(months=3)
        end   = row['_debut'] + pd.DateOffset(months=12)
        mask  = (google_trends_top3000['_date'] >= start) & (google_trends_top3000['_date'] <= end)
        window = google_trends_top3000[mask][['_date', row[gt_col_field]]].copy()
        window[row[gt_col_field]] = pd.to_numeric(window[row[gt_col_field]], errors='coerce')
        window['month'] = window['_date'].apply(
            lambda d: (d.year - row['_debut'].year) * 12 + (d.month - row['_debut'].month)
        )
        ax.plot(window['month'], window[row[gt_col_field]], marker='o', markersize=3,
                alpha=0.4, linewidth=1)

    ax.axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.set_xticks(range(-3, 13))
    ax.set_xticklabels([f"M{m:+d}" if m != 0 else "Debut" for m in range(-3, 13)])
    ax.set_xlabel('Months relative to first top-20 hit')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

# ── Step 5: 4 graphs ──────────────────────────────────────────────────────────

ohw       = first_top20[first_top20['is_one_hit_wonder']]
hitmakers = first_top20[~first_top20['is_one_hit_wonder']]

plot_trends(ohw,       'gt_col_web',     'Google Trends — web',     'Web: One-Hit Wonders (2005–2020)')
plot_trends(hitmakers, 'gt_col_web',     'Google Trends — web',     'Web: Hitmakers (2005–2020)')
plot_trends(ohw,       'gt_col_youtube', 'Google Trends — YouTube', 'YouTube: One-Hit Wonders (2005–2020)')
plot_trends(hitmakers, 'gt_col_youtube', 'Google Trends — YouTube', 'YouTube: Hitmakers (2005–2020)')


In [ ]:
# Compute and plot the average Google Trends value by month (M-3 to M+12)
# for one-hit wonders vs hitmakers, for both web and YouTube.
# Each line is the mean across all artists in that group at that relative month.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def collect_monthly_values(group_df, gt_col_field):
    records = []
    sub = group_df[group_df[gt_col_field].notna()]
    for _, row in sub.iterrows():
        start = row['_debut'] - pd.DateOffset(months=3)
        end   = row['_debut'] + pd.DateOffset(months=12)
        mask  = (google_trends_top3000['_date'] >= start) & (google_trends_top3000['_date'] <= end)
        window = google_trends_top3000[mask][['_date', row[gt_col_field]]].copy()
        window[row[gt_col_field]] = pd.to_numeric(window[row[gt_col_field]], errors='coerce')
        window['month'] = window['_date'].apply(
            lambda d: (d.year - row['_debut'].year) * 12 + (d.month - row['_debut'].month)
        )
        window['value'] = window[row[gt_col_field]]
        records.append(window[['month', 'value']])
    combined = pd.concat(records)
    return combined.groupby('month')['value'].mean()

ohw       = first_top20[first_top20['is_one_hit_wonder']]
hitmakers = first_top20[~first_top20['is_one_hit_wonder']]

for suffix, label in [('_web', 'Web'), ('_youtube', 'YouTube')]:
    gt_col = f'gt_col{suffix}'
    ohw_means  = collect_monthly_values(ohw,       gt_col)
    hit_means  = collect_monthly_values(hitmakers, gt_col)

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(ohw_means.index,  ohw_means.values,  marker='o', label='One-hit wonders')
    ax.plot(hit_means.index,  hit_means.values,  marker='o', label='Hitmakers')
    ax.axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.6)
    ax.set_xticks(range(-3, 13))
    ax.set_xticklabels([f"M{m:+d}" if m != 0 else "Debut" for m in range(-3, 13)])
    ax.set_xlabel('Months relative to first top-20 hit')
    ax.set_ylabel(f'Mean Google Trends — {label}')
    ax.set_title(f'Avg Google Trends ({label}): One-Hit Wonders vs Hitmakers (2005–2020)')
    ax.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Compute and plot average Google Trends value by month (M-3 to M+12)
# for one-hit wonders vs hitmakers, across three time periods.
# Two graphs (web, YouTube) x three subplots each (one per period).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def collect_monthly_values(group_df, gt_col_field):
    records = []
    sub = group_df[group_df[gt_col_field].notna()]
    for _, row in sub.iterrows():
        start = row['_debut'] - pd.DateOffset(months=3)
        end   = row['_debut'] + pd.DateOffset(months=12)
        mask  = (google_trends_top3000['_date'] >= start) & (google_trends_top3000['_date'] <= end)
        window = google_trends_top3000[mask][['_date', row[gt_col_field]]].copy()
        window[row[gt_col_field]] = pd.to_numeric(window[row[gt_col_field]], errors='coerce')
        window['month'] = window['_date'].apply(
            lambda d: (d.year - row['_debut'].year) * 12 + (d.month - row['_debut'].month)
        )
        window['value'] = window[row[gt_col_field]]
        records.append(window[['month', 'value']])
    if not records: return pd.Series(dtype=float)
    return pd.concat(records).groupby('month')['value'].mean()

PERIODS = [(2006, 2010), (2011, 2015), (2016, 2020)]

for suffix, label in [('_web', 'Web'), ('_youtube', 'YouTube')]:
    gt_col = f'gt_col{suffix}'
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
    fig.suptitle(f'Avg Google Trends ({label}): One-Hit Wonders vs Hitmakers', fontsize=13)

    for ax, (yr_start, yr_end) in zip(axes, PERIODS):
        period = first_top20[first_top20['first_charting_year'].between(yr_start, yr_end)]
        ohw       = period[period['is_one_hit_wonder']]
        hitmakers = period[~period['is_one_hit_wonder']]

        ohw_means = collect_monthly_values(ohw,       gt_col)
        hit_means = collect_monthly_values(hitmakers, gt_col)

        ax.plot(ohw_means.index, ohw_means.values, marker='o', label=f'One-hit wonders (n={len(ohw[ohw[gt_col].notna()])})')
        ax.plot(hit_means.index, hit_means.values, marker='o', label=f'Hitmakers (n={len(hitmakers[hitmakers[gt_col].notna()])})')
        ax.axvline(0, color='black', linestyle='--', linewidth=1.2, alpha=0.6)
        ax.set_xticks(range(-3, 13))
        ax.set_xticklabels([f"M{m:+d}" if m != 0 else "Debut" for m in range(-3, 13)], fontsize=7)
        ax.set_title(f'{yr_start}–{yr_end}')
        ax.set_xlabel('Months relative to first top-20 hit')
        ax.legend(fontsize=8)

    axes[0].set_ylabel(f'Mean Google Trends — {label}')
    plt.tight_layout()
    plt.show()
